# Lecture 3 — Structured Programming

## Checkers

In **Lab 2** you’ll build a **tic‑tac‑toe** game. In this lecture we’ll build a (simplified) **checkers** game, which is much more complex. The point isn’t to memorize every rule of checkers — it’s to practice **structured programming**: solving a big problem by breaking it into small, testable pieces.

As we work through the notebook, we’ll be weaving together a few threads:

- The main topic: **structured programming** (decomposition, functions, clear interfaces).
- The context: the checkers game we are building.
- Quick Python review: lists, dictionaries, loops, and functions.
- Workflows: how you actually build and debug a larger program.
- Lab connections: ideas you can reuse in your tic‑tac‑toe lab.

You won’t pick all of this up by reading once. Run the cells, experiment, and make sure you understand each function before moving on.

If you are unfamiliar with checkers, take a moment to skim the rules on Wikipedia: [Draughts](https://en.wikipedia.org/wiki/Draughts).

Why games — and why *these* games? Checkers, tic‑tac‑toe (and later blackjack) are games most of you already know how to play. That's deliberate. Because you already understand the *domain*, you can focus entirely on the real question — *how do I translate something I understand into code?* — instead of wrestling with an unfamiliar problem **and** an algorithm at the same time.

One more note on how this lecture works: it isn't a recipe ("step one, do this; step two, do that"). Along the way we'll regularly step aside to talk about the *process* of programming — how to think through a problem, how to start when you don't know what to do, and why there are usually several ways to do the same thing, some better, some worse, and some simply a matter of choice.

## Our Plan

The goal of working through building a checkers game in this lecture is to demonstrate how we can build something complicated by solving smaller problems. Each solution to a small problem will be implemented as a function. Once a function is worked out and tested, we can treat it as an **abstraction barrier**—we no longer need to worry about the details of that specific problem when building larger pieces.

Let’s break the problem down into smaller parts:

- **Game representation:**  
  We need a way to represent the game board and pieces in memory. Once this is defined, we can write a function that creates a game board with pieces in their starting positions.

- **Game play (moves):**  
  We need to take an existing game board and modify it according to a player’s desired move. Once this is worked out, we’ll have a function that takes a board and a move description and updates the board accordingly.

- **Game display:**  
  The in-memory representation of the game is not very human-friendly. We’ll write a function that takes a game board and displays it in a clear, readable way.

- **Player interaction:**  
  We need a convention for how players specify their moves in a way that is natural and easy for humans to use.

- **Overall game loop:**  
  To actually play the game, we’ll need to:
  - Create a new game board  
  - Draw the game board  
  - Ask the first player for a move  
  - Update the board  
  - Draw the game board again  
  - Ask the second player for a move  
  - Update the board  
  - Check whether a player has won  
  - If not, repeat the process starting with the first player

A useful way to see the structure of this plan: the board in memory *is* the current state of the game, and playing the game is nothing more than manipulating that representation with functions. Notice also that the display and the gameplay are **independent** problems — each interacts with the representation, but they never need to know about each other. We have *factorized* the big problem.

Once each piece is solved, the finished game is just a matter of calling the solved pieces in the right order, inside a loop.

### Game Representation

Checkers is typically played on the same 8×8 board as chess. To play the game, we need to store the state of the game—that is, the contents of each square—in memory. A natural way to represent the board is as a matrix, where each element corresponds to a square on the board.

We’ll use the following convention: a square is empty when the corresponding matrix element is `0`, and it contains a piece from player 1 or player 2 when the value is `1` or `2`, respectively.

We may decide to change this convention later, so rather than hard-wiring the values `0`, `1`, and `2` directly into our code, we’ll use variables to store these identifiers. This makes the code easier to read and easier to modify. Similarly, we’ll keep the board size flexible rather than fixing it permanently at 8×8.

A few words on this convention. It's exactly that — a convention, a choice we're making. We could just as well have used `-1`, `0`, and `+1`; nothing forces `1` and `2`. But once we implement a convention, we have to stick to it consistently.

This is also why we define **named constants** rather than sprinkling raw numbers through the code. If someone reading your code sees `if x == 1:` they have no idea what the `1` means. If they see `if player == player_1:` the *intention* is right there in the line. It's not just for other readers, either — choosing good names makes it easier for *you* to think about your own code, because it turns your thoughts into words.

The same reasoning applies to the board size. Checkers is always 8×8, but there is no reason to hardwire that. If we write everything in terms of a `size` variable, the same code will produce a 10×10 board for free — and a test like `x < size` documents its own intent in a way that `x < 8` never can. None of this is strictly *necessary* to make the code work; it makes the code better for everybody, including future you.

In [3]:
# Index assignment for the matrix representation of the game board
player_1 = 1
player_2 = 2
empty = 0

# Game board size
size = 8

We will represent the board as a **matrix**: a list of `size` rows, where each row is a list of `size` integers.

That means we need to *construct* a “list of lists”. There are lots of ways to do this in Python. We’ll start with a straightforward loop (easy to read), and then look at more compact shortcuts (which can be trickier than they look).

An obvious way to create such a list is with a loop:

It helps to read this loop from the inside out: the inner loop builds one **row** by appending `empty` to it `size` times; the outer loop then appends that finished row to the board, and repeats the whole thing `size` times.

(One notebook note: the bare `board` on the last line is there because Jupyter displays the value of the *last expression* evaluated in a cell — that's why the board shows up below the cell without an explicit `print`.)

In [4]:
board=list()
for i in range(size):
    row=list()
    for j in range(size):
        row.append(empty)
        
    board.append(row)

board

[[0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0]]

Our approach here is correct, but not necessarily the most compact. Python provides very terse syntax for achieving seemingly similar results... but it's not the same. 

For example, you can make a list of any length with exactly the same element using the following syntax:

In [5]:
[10]*8

[10, 10, 10, 10, 10, 10, 10, 10]

So we can make our matrix in one line:

In [6]:
board=[[empty]*size]*size
board

[[0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0]]

Unfortunately, there is a subtle problem here.

In this case, Python did **not** create `size` independent rows. Instead, it created **one** inner list and then put *multiple references to that same list* into the outer list. That means if you mutate one row, you mutate them all.

Notice what happens:

Why does Python behave this way? When Python "copies" a **number**, it really copies it — a number occupies a few bytes, so a copy is cheap. But a **list** could be gigabytes. So when Python replicates a list, it doesn't duplicate the contents; it hands out a new *reference* to the same list. `[10]*8` gave us eight independent copies of the number 10, but `[row]*size` gives us `size` references to *one* row.

This looks like a pedantic point, but I have been caught by this by accident more than once — it's the kind of bug where the code *looks* right and even *runs* right until it suddenly doesn't. It's worth developing the habit of thinking not just about what the syntax says, but about what is happening underneath. We'll come back to this when we cover `copy` and `deepcopy` in the next lecture.

In [7]:
board[1][1]=1
board

[[0, 1, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0]]

*Question:* Why didn't every element become 1?


*Answer:* Internally, Python stores references to lists, while it stores copies of numbers and strings.

### Aside: what we just saw has a name — *pointers*

The reason every row changed together is that a Python variable does not hold a list — it holds a **reference** to a list: an arrow pointing at one object in memory. `[[empty]*size]*size` made one inner list and eight arrows to it.

Older, closer-to-the-machine languages make the arrow itself an explicit thing you can hold and manipulate. In **C**, it is called a **pointer**, and it is literally a memory address — the same memory addresses we met last chapter when an instruction like `ADD addr8, addr9 -> addr12` named the *locations* of its operands rather than their values:

```c
int x = 5;      /* a box in memory holding 5        */
int *p = &x;    /* p holds the ADDRESS of x's box   */
*p = 7;         /* follow the arrow and write 7,    */
                /* so x is now 7                    */
```

Two operators do all the work: `&x` means "the address of `x`", and `*p` means "the thing `p` points at". Our repeated-row bug is, in C's vocabulary, eight pointers to one box. And C's two kinds of equality — `p == q` (same address?) versus `*p == *q` (equal contents?) — are exactly Python's `is` versus `==`.

Python never shows you the arrows, but they are there. `id()` reveals the address a name points at:

In [ ]:
a = [1, 2, 3]
b = a               # copies the ARROW, not the list
print(id(a), id(b)) # same address: one list, two names
b[0] = 99
a                   # "changing b" changed a — same box

Keep this mental model for the rest of the course: **assignment copies arrows, never boxes.** It explains the bug above, it explains in a moment why `move_piece` can update the board without returning it, and when we meet `copy` and `deepcopy` in the AI lecture, it will explain exactly what those buy us. We will come back to C again when we talk about loops.

So we'll take a hybrid approach:

In [8]:
board=list()
for i in range(size):
    board.append([empty]*size)

board[1][1]=1
board

[[0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0]]

In [9]:
board[0][0]=1
board

[[1, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0]]

### Aside: test as you go

Notice the pattern in this notebook: every time we write a piece of code, we immediately run a quick test or two. You cannot trust an abstraction barrier until you've tested it. If you write a function, skip the test, use it inside a second function, and *that* one crashes — you have no idea which layer went wrong, and you end up doing the testing anyway, later and under worse conditions.

A test doesn't have to be elaborate. You have some conception of what the code is supposed to do; set up a small situation that demonstrates it actually does that. Frequently the act of testing teaches you something that makes the *next* function you write better.

List comprehensions are a compact way of expressing certain loops. We will cover this topic later in the course. Here’s an example of creating an empty board in one line:

Don't worry if this syntax doesn't fully click yet — there's a whole lecture on comprehensions coming later. For now, just recognize that the one line below is functionally equivalent to the loop we wrote above.

In [10]:
board=[[empty]*size for i in range(size)]

board[1][1]=1
board

[[0, 0, 0, 0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0]]

Now let's write a function that will create a new game board for us with the pieces in the right spot.

How do we figure out where the pieces go? Look at an actual board and hunt for the pattern. (If you search the web for images of checkers setups, be a little careful — some of the pictures out there are set up *incorrectly*. The pieces always sit on the same-color squares.) Here is the starting position we're aiming for:

![Checkers starting position](figures/checkers_start.png)

Staring at a correct setup, a couple of observations emerge:

- Pieces occupy only alternating squares: on even rows they sit in one set of columns, on odd rows in the other.
- Only the first three and last three rows are filled — player 1's pieces at one end, player 2's at the other.

You might look at the same board and see a *different* pattern, and code it up a different way — that's fine, and it might even be a better way. When you're faced with something to code and you don't know where to start: go look at the thing and think about it logically. Ask yourself how you would explain the pattern to another person who couldn't see the board. Communicating it to the computer is the same act, just written in code.

Read through the code below carefully. Note how negative list indexing in Python (counting from the end of a list) lets us fill the *bottom* rows without knowing the board size — if we had hard-coded rows 5, 6, and 7, the function would only work for an 8×8 board.

In [11]:
def make_game_board(size=8):
    # Make an empty board
    board=[[empty]*size for i in range(size)]
    
    # Even Columns
    for i in range(0,size,2):
        board[1][i]=player_1
        board[-1][i]=player_2
        board[-3][i]=player_2
        
    # Odd Columns
    for i in range(1,size,2):
        board[0][i]=player_1
        board[2][i]=player_1
        board[-2][i]=player_2
    
    return board

Let's test our code:

In [12]:
board_0=make_game_board()
board_0

[[0, 1, 0, 1, 0, 1, 0, 1],
 [1, 0, 1, 0, 1, 0, 1, 0],
 [0, 1, 0, 1, 0, 1, 0, 1],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [2, 0, 2, 0, 2, 0, 2, 0],
 [0, 2, 0, 2, 0, 2, 0, 2],
 [2, 0, 2, 0, 2, 0, 2, 0]]

Looks good.

### Game Rules

#### Moves

We note that checkers is a simple game where each (non-king) piece moves forward and simultaneously to the left or right. So we can specify a move by choosing a piece and picking left or right. A move is simply taking a piece at location `x,y` to a location `x+x_offset,y+y_offset`, where the offsets depend on the player and the choosen direction. So the first step  moving a piece is to determine what is the appropriate offset for given the color (aka player number) and the direction of intended move (right or left).

There are many ways to represent this logic. For example we can write a function to compute the offset:

In [19]:
left_move=0
right_move=1

def player_moves(player,direction):
    if player==player_1:
        if direction==left_move:
            return (1,1)
        elif direction==right_move:
            return (1,-1)
    if player==player_2:
        if direction==left_move:
            return (-1,-1)
        elif direction==right_move:
            return (-1,1)


This works, but notice that the function above is basically just a lookup table: it maps `(player, direction)` to an offset.

When a “function” is only doing a fixed mapping like this, it’s often clearer (and more flexible) to store the mapping directly in a data structure, like a list or dictionary:

(As a bonus, looking a value up in a data structure is technically cheaper than calling a function — though at this scale the difference is irrelevant. The real reason to prefer the data structure is clarity.)

In [20]:
player_1_left_move=(1,1)
player_1_right_move=(1,-1)

player_2_left_move=(-1,-1)
player_2_right_move=(-1,1)

moves = [[player_1_left_move, player_1_right_move],
         [player_2_left_move, player_2_right_move]]

moves

[[(1, 1), (1, -1)], [(-1, -1), (-1, 1)]]

So we can recall a move simply by:

In [21]:
moves[player_1-1][right_move]

(1, -1)

The problem with using a list like `moves[player_1-1]` is that we start to **lose the abstraction** of what `player_1` and `player_2` mean.

Lists are indexed by **integers starting at 0**, so we have to manually make sure our player IDs line up with list indices (or remember to subtract 1 everywhere). That’s easy to get wrong and makes the code harder to read.

A more elegant solution is to use a **dictionary of dictionaries**:

Concretely: if we later decided to redefine `player_1 = 100`, the list version would break immediately — we'd be asking for `moves[99]`. The dictionary version below keeps working, because dictionary keys don't care what the values are, only that they're consistent.

In [22]:
moves={ player_1: {left_move: player_1_left_move, 
                   right_move:player_1_right_move},
        player_2: {left_move: player_2_left_move, 
                   right_move: player_2_right_move}}

moves

{1: {0: (1, 1), 1: (1, -1)}, 2: {0: (-1, -1), 1: (-1, 1)}}

In [23]:
moves[player_1][right_move]

(1, -1)

This `moves` dictionary is a much more elegant and flexible solution. We can change the definitions of `player_1`, `player_2`, `right_move`, and `left_move` without having to change the definition of `moves`. The message to walk away from this small discussion is that there sometimes are various ways to represent logic. In this case as a function, list of lists, or dictionary of dictionaries. And some provide more flexibility and/or are more elegant than others.

### Moving Pieces

Next let's code up a function that will take a board, player, location of a piece, and desired move and then returns a new board with the desired moved made. This function is going to be the heart of the game in a sense. It captures the rules of what players can and cannot do.

*Quick side:* We are going to put in some print statements so we see what is happening, but we don't necessarily want to see these statements all of the time, for example later we may wish to write an artificial intelligence that uses the same function to think about the game, and we  don't necessarily see everything it's thinking. So let's write a function that we can use for printing:

This is an extremely common pattern in real software. Production code — especially scientific code — typically uses a *logging* system with message levels: in debug mode everything gets spit out, in verbose mode you get detail, in quiet mode almost nothing, and you flip a switch to choose. Our one-argument `verbose` flag is the simplest possible version of that idea.

And there's a concrete reason we need it *now*: in the next lecture, a computer player will call `move_piece` over and over again just to ask "is this move possible?" while it thinks. We don't want a printout for every move it merely *considers*.

In [24]:
def print_message(message,verbose=True):
    if verbose:
        print(message)

In [25]:
print_message("Hello")

Hello


In [26]:
print_message("Hello",False)

Now to the heart of the game, a `move_piece` function that will apply game rules to move pieces. As arguments, this function will need:

* `board`: The game board
* `player`: Which player is making the move
* `location`: Where the game piece is located on the board
* `direction`: Which way to move to piece

Now let's think through the logic. The function should:

* Check if player's piece is at location
* Fetch the offset for the move
* Make sure the move (and possible jump move) is on the board
* If the target space empty, move the piece by emptying the starting position placing the piece into the target
* If the target space is filled with opponent's piece, but the next space along the diagonal is empty, empty the start, remove the target, and place piece next along diagonal.

Finally, we recognize that moves may be invalid and that we may need to convey this information back to the user. So we adopt a convention that if a move is invalid, the function returns `False`, otherwise it returns `True`. Later, this convention allows a AI or computer will know if a move is valid.

Before writing a function like this, it's worth taking a second — on paper, or in comments — to write down what the inputs are and what needs to happen. Think the logic through in *words* first, then turn the words into code. That is exactly what the bullet list above is.

Two small syntax notes about the code below:

- The boundary test is a chain of comparisons joined with `and`. Each comparison is one logical check, and the move is possible only if *all* of them are true. We could have written a stack of `if` statements instead; the `and` chain just says the same thing more compactly.
- You'll see a backslash `\` at the end of a line — it tells Python (and the reader) that the statement continues on the next line.

In [27]:
def move_piece(board,player,location,move,verbose=True):
    x,y=location
    
    # Check if player's piece is at location
    if not board[x][y] == player:
        print_message("Player does not have piece at location.",verbose)
        return False

    # Fetch the offset for the move
    x_offset,y_offset = moves[player][move]
    
    # Make sure the move is on the board:
    move_possible= x+x_offset < size and \
                    x+x_offset >= 0 and \
                    y+y_offset < size and \
                    y+y_offset >= 0
                
                
    jump_possible= x+2*x_offset < size and \
                    x+2*x_offset >= 0 and \
                    y+2*y_offset < size and \
                    y+2*y_offset >= 0
    
    if not (move_possible or jump_possible):
        print_message("Move is off of board.",verbose)
        return False
        
    # Try the move
    # Is the target space empty
    if move_possible and \
        board[x+x_offset][y+y_offset]==empty:
    
        # Make the move
        # Empty the spot
        board[x][y]=empty
        # Place player in new spot
        board[x+x_offset][y+y_offset]=player
        print_message("Moved.",verbose)            

        return True

    # Does the target space have an opponent's piece, and the space after empty
    elif jump_possible and \
            board[x+x_offset][y+y_offset]!=player and \
            board[x+2*x_offset][y+2*y_offset]==empty:

        # Make the move
        # Empty the spot
        board[x][y]=empty
        # Remove the opponent's piece
        board[x+x_offset][y+y_offset]=empty
        # Move player to new spot
        board[x+2*x_offset][y+2*y_offset]=player
        print_message("Took opponent's piece.",verbose)
        
        return True
    else:
        print_message("Move not possible.",verbose)
        return False


One subtlety worth pausing on: `move_piece` returns `True` or `False` — it never returns the board. So how does the board get updated? Because a list is passed **by reference**: the `board` parameter inside the function refers to the very same list-of-lists the caller passed in, so modifying it modifies the caller's board. If we had passed in a plain number and reassigned it inside the function, the caller's variable would be untouched. This is the same numbers-versus-lists distinction that bit us when we built the board with `*`.

Note also that we put everything in a single function. In principle, we could have broken the code up into several smaller functions — but there isn't much reuse of the pieces here, and scattering the logic can obfuscate the coder's intent. Having *all* the rules of the game in one well-documented place is itself a kind of clarity. This is a choice, not a law: real-world design is a constant judgment call about how much structure a problem deserves. And once this function works, it becomes an abstraction — the problem is solved, and we never have to think about the details again.

Let's test our code:

In [28]:
board_0=make_game_board()
board_0

[[0, 1, 0, 1, 0, 1, 0, 1],
 [1, 0, 1, 0, 1, 0, 1, 0],
 [0, 1, 0, 1, 0, 1, 0, 1],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [2, 0, 2, 0, 2, 0, 2, 0],
 [0, 2, 0, 2, 0, 2, 0, 2],
 [2, 0, 2, 0, 2, 0, 2, 0]]

In [29]:
move_piece(board_0,player_1,(2,1),left_move)
board_0

Moved.


[[0, 1, 0, 1, 0, 1, 0, 1],
 [1, 0, 1, 0, 1, 0, 1, 0],
 [0, 0, 0, 1, 0, 1, 0, 1],
 [0, 0, 1, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [2, 0, 2, 0, 2, 0, 2, 0],
 [0, 2, 0, 2, 0, 2, 0, 2],
 [2, 0, 2, 0, 2, 0, 2, 0]]

In [30]:
move_piece(board_0,player_2,(5,4),left_move)
board_0

Moved.


[[0, 1, 0, 1, 0, 1, 0, 1],
 [1, 0, 1, 0, 1, 0, 1, 0],
 [0, 0, 0, 1, 0, 1, 0, 1],
 [0, 0, 1, 0, 0, 0, 0, 0],
 [0, 0, 0, 2, 0, 0, 0, 0],
 [2, 0, 2, 0, 0, 0, 2, 0],
 [0, 2, 0, 2, 0, 2, 0, 2],
 [2, 0, 2, 0, 2, 0, 2, 0]]

Now let's test taking a piece:

In [31]:
move_piece(board_0,player_1,(3,2),left_move)
board_0

Took opponent's piece.


[[0, 1, 0, 1, 0, 1, 0, 1],
 [1, 0, 1, 0, 1, 0, 1, 0],
 [0, 0, 0, 1, 0, 1, 0, 1],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [2, 0, 2, 0, 1, 0, 2, 0],
 [0, 2, 0, 2, 0, 2, 0, 2],
 [2, 0, 2, 0, 2, 0, 2, 0]]

In [32]:
move_piece(board_0,player_1,(2,7),left_move)
board_0

Move is off of board.


[[0, 1, 0, 1, 0, 1, 0, 1],
 [1, 0, 1, 0, 1, 0, 1, 0],
 [0, 0, 0, 1, 0, 1, 0, 1],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0],
 [2, 0, 2, 0, 1, 0, 2, 0],
 [0, 2, 0, 2, 0, 2, 0, 2],
 [2, 0, 2, 0, 2, 0, 2, 0]]

At this point we have the core *data* and *rules* we need for a basic version of checkers.

## User Friendly Output

Next, we’ll improve the **output** so a human can actually read the board state.


In [33]:
player_1_piece="X"
player_2_piece="O"
empty_space=" "

Next, let's define a function (or mapping) that converts the internal numeric representation into the character we want to display:

In [34]:
def space_character(player):
    if player==player_1:
        return player_1_piece
    elif player==player_2:
        return player_2_piece
    else:
        return empty_space

Actually, recalling earlier, we can do this much nicer with a dictionary:

In [35]:
space_character= { player_1: player_1_piece,
                   player_2: player_2_piece,
                   empty: empty_space }

space_character

{1: 'X', 2: 'O', 0: ' '}

In [37]:
space_character[empty]

' '

Now we can print our board by looping over its matrix representation and using `print` to write the correct characters in the right spot.

Note the `end=" "` argument to `print` keeps the cursor from going to the next line when we don't want it to do so. We'll use an empty `print` to get to the next line when we do.

In [40]:
def draw_board(board):
    for i in range(size):
        for j in range(size):
            print(space_character[board[i][j]],end=" ")
        print()

In [41]:
draw_board(board_0)

  X   X   X   X 
X   X   X   X   
      X   X   X 
                
                
O   O   X   O   
  O   O   O   O 
O   O   O   O   


It'll be hard for a player to determine the position of the pieces by index, so lets adopt a scheme where the columns are specified by a letter and the row by a number. Let's also change our `draw_board` to put this information on the board. 

In [42]:
# Row labels (A, B, C, ...) — use only as many as we need for this board size
row_names = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[:size]
row_map = dict(zip(row_names, range(size)))

row_map

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7}

How did we do this? Let's take this step by step:

In [43]:
{"A":1, "B":2}

{'A': 1, 'B': 2}

In [44]:
dict([("A",1),("B",2)])

{'A': 1, 'B': 2}

In [45]:
list(range(size))

[0, 1, 2, 3, 4, 5, 6, 7]

In [46]:
list(zip(row_names,range(size)))

[('A', 0),
 ('B', 1),
 ('C', 2),
 ('D', 3),
 ('E', 4),
 ('F', 5),
 ('G', 6),
 ('H', 7)]

In [47]:
dict(zip(row_names,range(size)))

{'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7}

One more time, let's break it down:

Two things to notice here. `zip` takes two lists and interleaves them — first element with first element, second with second, each pair becoming a tuple. Think of it as shuffling two decks of cards together.

Also note that we wrap `range(...)` in `list(...)`. On its own, `range` doesn't build the list immediately — it produces values lazily, on demand. We'll talk about lazy evaluation later in the course; for now, `list(range(n))` forces the actual list into existence so we can look at it.

In [48]:
print(row_names)
print(list(range(size)))
print(list(zip(row_names,range(size))))

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
[0, 1, 2, 3, 4, 5, 6, 7]
[('A', 0), ('B', 1), ('C', 2), ('D', 3), ('E', 4), ('F', 5), ('G', 6), ('H', 7)]


Note the use of zip. Let's do the same for the columns. Here we'll use map to turn a list of numbers to a list of strings:

(`map` applies a function to every element of a list and hands back the results — an easy way to convert a list of things from one type to another.)

Why show these idioms at all? There are a dozen ways to do the same thing in Python, and these particular tricks — `zip`, `map`, building dictionaries from pairs — are ones you will constantly encounter in other people's code. Part of becoming a data scientist is building a little Python interpreter in your head: you need to be able to *read* these constructions even before you start writing them yourself.

In [49]:
column_names=list(map(str,range(1,size+1)))
column_map=dict(zip(column_names,range(size)))

column_map

{'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7}

Let's break this one down:

In [50]:
list(range(1,size+1))

[1, 2, 3, 4, 5, 6, 7, 8]

In [51]:
list(map(str,range(1,size+1)))

['1', '2', '3', '4', '5', '6', '7', '8']

Now let's draw our new board:

In [56]:
def draw_board(board):
    print(" ",end=" ")
    for j in range(size):
        print(column_names[j],end=" ")
    print()
    
    for i in range(size):
        print(row_names[i],end=" ")
        for j in range(size):
            print(space_character[board[i][j]],end=" ")
        print()

In [57]:
draw_board(board_0)

  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C   X   X   X   X 
D                 
E                 
F O   O   O   O   
G   O   O   O   O 
H O   O   O   O   


Much nicer. 

But now the user will give us locations like "C3" and we'll have to convert it to a pair of indexes for the matrix. These are stored in the `row_map` and `column_map` dictionaries, but we can't trust the user to correctly supply an input, so lets be careful by checking the type of input, length, and making sure it's upper case:

Notice what we're doing at a higher level: the user-facing layer (letters, numbers, strings like `"C3"`) is completely separate from the memory representation (matrix indices). The interface is a solved problem, the game logic is a solved problem, and they connect only through these small translation functions.

In [58]:
def parse_location(l_string):
    if not isinstance(l_string,str):
        print_message("Bad Input. Location must be string.")
        return False
    
    if len(l_string)!=2:
        print_message("Bad Input. Location must be 2 characters.")
        return False
    
    row=l_string[0].upper()
    col=l_string[1].upper()
    
    if row not in row_map:
        print_message("Bad Row.")
        return False

    if col not in column_map:
        print_message("Bad Column.")
        return False

    return row_map[row],column_map[col]
    

Test our code:

In [59]:
parse_location("C4")

(2, 3)

Similarly, we'll set things up so the user can specify `"L"` or `"R"` for moves (left or right).

In [60]:
def parse_move(m_string):
    """Convert a move direction like 'L' or 'R' into our internal constants."""
    if not isinstance(m_string, str):
        print_message("Bad input. Move must be a string.")
        return -1

    m_string = m_string.strip()
    if len(m_string) != 1:
        print_message("Bad input. Move must be 1 character: 'L' or 'R'.")
        return -1

    if m_string.upper() == "L":
        return left_move

    if m_string.upper() == "R":
        return right_move

    print_message("Bad move. Move must be 'L' or 'R'.")
    return -1


Finally, let's put it all together:

In [61]:
def nice_move_piece(board,player,location,move):
    loc=parse_location(location)
    mov=parse_move(move)

    if loc and mov!=-1:
        return move_piece(board,player,loc,mov)
    else:
        print_message("Bad move.")
        return False 

... and test again:

In [62]:
board_0=make_game_board()
draw_board(board_0)
nice_move_piece(board_0,player_1,"C4","L")
draw_board(board_0)

  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C   X   X   X   X 
D                 
E                 
F O   O   O   O   
G   O   O   O   O 
H O   O   O   O   
Moved.
  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C   X       X   X 
D         X       
E                 
F O   O   O   O   
G   O   O   O   O 
H O   O   O   O   


## A Game Program

Next, we don’t want the player to be making Python function calls directly.

Instead, the game should:

1. Show the current board.
2. Ask the current player for a location and a direction.
3. Apply the move (or reject it and ask again).
4. Repeat until someone wins.

We’ll use Python’s `input()` function to interact with the player:

Because we built abstractions along the way, the loop below reads almost like English: *keep asking for moves until you get a good one*. If all the parsing and validity-checking logic were dumped inline here instead, the intent would be buried.

In [63]:
def take_move(board, player):
    """Prompt the user until they enter a valid move."""
    good_move = False

    while not good_move:
        loc_str = input("Input location (e.g., C4): ")
        mov_str = input("Input move (L/R): ")

        good_move = nice_move_piece(board, player, loc_str, mov_str)


Let's test this function with a single move before moving on.

Example input to try:

- Location: `"C4"`
- Move: `"L"`

If you ever get stuck at an input prompt, you can stop the cell with **Kernel → Interrupt** (or the stop button in most notebook interfaces).

In [64]:
board_0 = make_game_board()
draw_board(board_0)

# OPTIONAL (interactive): make one move using input()
RUN_INTERACTIVE_DEMO = False  # set to True to enable

if RUN_INTERACTIVE_DEMO:
    take_move(board_0, player_1)
    draw_board(board_0)


  1 2 3 4 5 6 7 8 
A   X   X   X   X 
B X   X   X   X   
C   X   X   X   X 
D                 
E                 
F O   O   O   O   
G   O   O   O   O 
H O   O   O   O   


Now let's write a function that determines if the game is won, so it can stop:

In [67]:
# Function to count the number of pieces of a specific player
def count_pieces(board,player):
    n=0
    for i in range(size):
        for j in range(size):
            if board[i][j]==player:
                n+=1                
    return n


def game_won(board):
    player_1_n=count_pieces(board,player_1)
    player_2_n=count_pieces(board,player_2)

    if player_1_n==0:
        return player_2
    if player_2_n==0:
        return player_1

    return False


Pulling all of these functions together, we get a basic checkers game.

Notes / limitations of this simplified version:

- It does **not** handle king pieces.
- It does not enforce “must capture” rules.
- It will not detect stalemates (no legal moves).

Even with these limitations, it demonstrates how breaking a problem into small functions makes a larger program manageable.


In [68]:
def checkers_game():
    
    print ("Welcome to Checkers.")
    print ("--------------------")

    # Make a game board
    board_0=make_game_board()
    
    # Start with player 1
    player=player_1
    
    this_game_won=False
    while not this_game_won:
        # Draw the board
        draw_board(board_0)
        
        # Make a move
        print("Player",player,"move:")
        take_move(board_0,player)

        # Check if the game has been won
        this_game_won=game_won(board_0)

        # Switch players
        if player==player_1:
            player=player_2
        else:
            player=player_1
            
        
    print("Winner is player:",this_game_won)
          

In [ ]:
# OPTIONAL (interactive): play a full human-vs-human game
RUN_FULL_GAME = False  # set to True to enable

if RUN_FULL_GAME:
    checkers_game()


## Summary

In this lecture we practiced **structured programming** by building a small (simplified) checkers program from scratch.

Key takeaways:

- **Decompose** a big problem into smaller functions, and test each function as you go.
- Use clear **data representations** (here: an 8×8 matrix) and named constants (`empty`, `player_1`, `player_2`) to make code readable.
- Be careful with Python shortcuts like `[[x]*n]*n`: they can create **shared references**. Prefer a loop or a list comprehension for nested lists.
- Store fixed mappings (like move offsets) in **data structures** (dictionaries) instead of hard‑coding logic everywhere.
- Write helper functions to improve usability:
  - a `draw_board` function for readable output
  - input parsing (`parse_location`, `parse_move`) to validate and convert user input
- A “game program” is usually just a **loop** that repeatedly:
  1. shows state,
  2. gets input,
  3. updates state,
  4. checks for termination.

Next time (Lecture 4), we’ll package this code into a module and use recursion to start exploring a simple AI player.
